## **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
if IS_COLAB or IS_KAGGLE:
    !pip install optuna

import optuna

In [4]:
from Challenge import paths

Running on local — storage at: /home/luigi/RecSys


## **Load Data**

In [5]:
# Load datasets
folds = paths.load_cv_folds(k=5)

URM_train, URM_val = folds[0]

## **Test Different evaluation func**

In [6]:
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
item_knn = ItemKNNCFRecommender(URM_train)
item_knn.fit()

Similarity column 6969 (100.0%), 4741.11 column/sec. Elapsed time 1.47 sec


## **Builtin Evaluator**

In [7]:
from Evaluation.Evaluator import EvaluatorHoldout
evaluator = EvaluatorHoldout(URM_val, [20])

EvaluatorHoldout: Ignoring 40 ( 0.1%) Users that have less than 1 test interactions


In [8]:
%%time
evaluator.evaluateRecommender(item_knn)[0].loc[20, 'RECALL']

EvaluatorHoldout: Processed 27055 (100.0%) in 6.09 sec. Users per second: 4442
CPU times: user 5.87 s, sys: 201 ms, total: 6.07 s
Wall time: 6.09 s


0.20954323309472742

### **Simple loop**

In [9]:
import numpy as np

def evaluate_recommender(recommender, at, URM_validation):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float64) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

In [10]:
%%time
evaluate_recommender(item_knn, at=20, URM_validation=URM_val)

CPU times: user 5.19 s, sys: 8.86 ms, total: 5.2 s
Wall time: 5.23 s


np.float64(0.20954323309472742)

## **Evaluate user_id in parallel**

In [11]:
from joblib import Parallel, delayed

def evaluate_single_user(user_id, recommender, at, relevant_items):
    """
    Worker function to evaluate a single user.
    """
    if len(relevant_items) == 0:
        return 0.0, 0

    # This line is where the magic (or bottleneck) happens
    recommended_items = recommender.recommend(user_id, cutoff=at)
    
    is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
    recall_score = np.sum(is_relevant, dtype=np.float64) / relevant_items.shape[0]
    
    return recall_score, 1

def evaluate_recommender_parallel(recommender, at, URM_validation, n_jobs=-1):
    """
    n_jobs=-1 uses all available CPU cores.
    """
    user_ids = np.arange(URM_validation.shape[0])
    
    # We prepare the arguments generator
    # Note: accessing URM rows is fast, but passing them to processes has overhead
    # We use a generator to be memory efficient
    tasks = (
        delayed(evaluate_single_user)(
            user_id, 
            recommender, 
            at, 
            URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        ) 
        for user_id in user_ids
    )
    
    # Run in parallel
    # 'prefer="processes"' forces it to bypass the GIL
    results = Parallel(n_jobs=n_jobs, prefer="processes")(tasks)
    
    # Aggregate results
    total_recall = sum(r[0] for r in results)
    num_eval = sum(r[1] for r in results)
    
    if num_eval == 0:
        return 0.0
        
    return total_recall / num_eval

In [12]:
%%time
evaluate_recommender_parallel(item_knn, at=20, URM_validation=URM_val)

CPU times: user 1.28 s, sys: 78.4 ms, total: 1.36 s
Wall time: 2.42 s


np.float64(0.20954323309472742)

## **Evaluate in batches**

In [13]:
def evaluate_recommender_batch(recommender, at, URM_validation, batch_size=1000):
    cumulative_recall = 0.0
    num_eval = 0
    
    # We iterate through all users in steps of 'batch_size'
    num_users = URM_validation.shape[0]
    
    for start_idx in range(0, num_users, batch_size):
        end_idx = min(start_idx + batch_size, num_users)
        user_batch = list(range(start_idx, end_idx))
        
        # 1. Get recommendations for the WHOLE batch at once
        # This is where the massive speedup comes from
        # Expected shape: List of lists or (batch_size, at) array
        batch_recommendations = recommender.recommend(user_batch, cutoff=at)
        
        # 2. Calculate metrics for this batch
        # We still loop here, but it's fast because the heavy prediction is done
        for i, user_id in enumerate(user_batch):
            # Get ground truth items from sparse matrix
            start_pos = URM_validation.indptr[user_id]
            end_pos = URM_validation.indptr[user_id+1]
            relevant_items = URM_validation.indices[start_pos:end_pos]
            
            if len(relevant_items) > 0:
                num_eval += 1
                
                # Get the specific recommendations for this user
                # (Assuming batch_recommendations is a list or matrix indexed by i)
                recommended_items = batch_recommendations[i]
                
                # Calculate Recall
                is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
                recall_score = np.sum(is_relevant, dtype=np.float64) / relevant_items.shape[0]
                cumulative_recall += recall_score

    if num_eval == 0:
        return 0.0
        
    return cumulative_recall / num_eval

In [14]:
%%time
evaluate_recommender_batch(item_knn, at=20, URM_validation=URM_val)

CPU times: user 3.02 s, sys: 122 ms, total: 3.14 s
Wall time: 3.15 s


np.float64(0.20954323309472742)

## **Evaluate batches in parallel**

In [15]:
def _worker_evaluate_subset(user_subset, recommender, at, URM_validation, batch_size):
    """
    This function runs on a separate CPU core.
    It processes a specific subset of users in batches.
    """
    cumulative_recall = 0.0
    num_eval = 0
    
    # Loop over the subset in chunks (batches)
    for i in range(0, len(user_subset), batch_size):
        # Get the chunk of user_ids
        batch_users = user_subset[i : i + batch_size]
        
        # 1. VECTORIZED PREDICTION (The fast part)
        batch_recommendations = recommender.recommend(batch_users, cutoff=at)
        
        # 2. METRIC CALCULATION
        for j, user_id in enumerate(batch_users):
            start_pos = URM_validation.indptr[user_id]
            end_pos = URM_validation.indptr[user_id+1]
            relevant_items = URM_validation.indices[start_pos:end_pos]
            
            if len(relevant_items) > 0:
                num_eval += 1
                recommended_items = batch_recommendations[j]
                
                is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
                recall_score = np.sum(is_relevant, dtype=np.float64) / relevant_items.shape[0]
                cumulative_recall += recall_score
                
    return cumulative_recall, num_eval

def evaluate_recommender_hybrid(recommender, at, URM_validation, n_jobs=-1, batch_size=1000):
    """
    n_jobs: Number of CPU cores (-1 = all cores)
    batch_size: How many users to predict at once inside each core
    """
    users_to_eval = np.arange(URM_validation.shape[0])
    
    # 1. Split users into N chunks (one for each CPU core)
    # If n_jobs is -1, joblib detects the number of cores automatically.
    # We use a helper to determine effective n_jobs for splitting
    from joblib import cpu_count
    effective_jobs = cpu_count() if n_jobs == -1 else n_jobs
    
    # Split the user array into roughly equal parts
    user_splits = np.array_split(users_to_eval, effective_jobs)

    # 2. Dispatch to Joblib
    # prefer="processes" is essential to bypass the Python GIL
    results = Parallel(n_jobs=n_jobs, prefer="processes")(
        delayed(_worker_evaluate_subset)(
            split, 
            recommender, 
            at, 
            URM_validation, 
            batch_size
        ) 
        for split in user_splits
    )
    
    # 3. Aggregate results from all cores
    total_recall = sum(r[0] for r in results)
    total_eval = sum(r[1] for r in results)
    
    if total_eval == 0:
        return 0.0
        
    return total_recall / total_eval

In [16]:
%%time
evaluate_recommender_hybrid(item_knn, at=20, URM_validation=URM_val)

CPU times: user 29.5 ms, sys: 13 ms, total: 42.4 ms
Wall time: 914 ms


np.float64(0.20954323309473646)

## **Results**
Best one is to precess batches in parallel.

- From 6.2 second of the builtin evaluator to 0.9 for itemKNN
(Wrong after the 14th decimal digit, not a problem)